# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sujan-lab-cell/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Unit of Analysis + Time Window

The unit of analysis for this project is one webpage (content item). Each row represents a single webpage and its associated performance data.

For this assignment, I will use a mid-panel month (March 2026) for analysis and feature creation. I am not using the final month because it should be reserved for evaluation and to avoid using future information.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Features
I will use historical webpage performance signals such as search volume, CTR, average position, engagement rate, scroll rate, word count, and content type.

### Label
The label is the proxy target for whether a webpage should be prioritized for content refresh.

### Context
Context fields describe the webpage, such as content ID, client ID, main intent, and access/profile information. These help identify and describe the data but are not the prediction target.

### Excluded
I will exclude identifier fields, future information, and any label-derived columns because they can cause leakage or do not help the model generalize.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [21]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT COUNT(*)
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

The unit of analysis is one webpage (content item) for one client on one report date.

The sample output confirms that each row represents a single daily performance record for one webpage.

In [22]:
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 5
""")

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_clau

In [23]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬────────────┐
│ total_rows │ start_date │  end_date  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘

In [25]:
con.sql(f"""
SELECT *
FROM glob('{rel}/**/*.parquet')
LIMIT 50
""")

┌────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                                  file                                                  │
│                                                varchar                                                 │
├────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet                                         │
│ hf://datasets/FlyRank/internship-warehouse/dim_content.parquet                                         │
│ hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet │
│ hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet │
│ hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet │
│ hf://datasets/FlyRank/internship-wa

In [26]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 1
""")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Data Limits

This dataset contains anonymized search and webpage performance data. It does not include the actual webpage content, user feedback, competitor information, seasonal events, or Google algorithm updates.

The model can identify patterns in historical performance, but it cannot explain the exact reason why a webpage gained or lost traffic. Therefore, its predictions should be used as decision support rather than as a final decision.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.